# Travel style classifier — synthetic destinations

Generates 150 labeled rows, compares three sklearn classifiers with `GridSearchCV`, and exports `travel_classifier_final.joblib`, `preprocessor.joblib`, and `label_encoder.joblib` under `backend/ml/models/` (paths used by the live classifier tool).

In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

random.seed(42)
np.random.seed(42)

STYLES = ["Adventure", "Relaxation", "Culture", "Budget", "Luxury", "Family"]
CLIMATES = ["tropical", "temperate", "cold", "desert", "mediterranean"]

rows = []
for i in range(180):
    style = random.choice(STYLES)
    budget = float(random.randint(40, 450))
    climate = random.choice(CLIMATES)
    hiking = random.random() * 10
    beach = random.random() * 10
    culture = random.random() * 10
    if style == "Adventure":
        hiking += random.uniform(3, 7)
    if style == "Relaxation":
        beach += random.uniform(3, 7)
    if style == "Culture":
        culture += random.uniform(3, 7)
    rows.append(
        {
            "destination_city": f"City_{i}",
            "country": random.choice(["A", "B", "C", "D"]),
            "travel_style": style,
            "avg_annual_temp_c": random.uniform(-5, 32),
            "seasonal_range_c": random.uniform(5, 25),
            "cost_per_day_avg_usd": budget,
            "meal_budget_usd": budget * 0.25,
            "hotel_night_avg_usd": budget * 0.55,
            "flight_cost_usd": random.uniform(200, 1200),
            "museum_count": random.uniform(0, 120),
            "monument_count": random.uniform(0, 80),
            "festival_score": random.random() * 10,
            "beach_score": beach,
            "scenic_score": random.random() * 10,
            "wellness_score": random.random() * 10,
            "culture_score": culture,
            "hiking_score": hiking,
            "nightlife_score": random.random() * 10,
            "family_score": random.random() * 10,
            "luxury_score": random.random() * 10,
            "safety_score": random.uniform(4, 10),
            "tourist_density_score": random.random() * 10,
            "adventure_sports_score": random.random() * 10,
            "near_mountains": random.choice([0.0, 1.0]),
            "near_beach": random.choice([0.0, 1.0]),
            "english_friendly_score": random.uniform(3, 10),
            "public_transport_score": random.uniform(3, 10),
            "latitude": random.uniform(-40, 60),
            "longitude": random.uniform(-120, 120),
            "climate_bucket": climate,
        }
    )

df = pd.DataFrame(rows)
feature_cols = [c for c in df.columns if c not in ("destination_city", "country", "travel_style")]
num_cols = [c for c in feature_cols if c != "climate_bucket"]
cat_cols = ["climate_bucket"]

X = df[feature_cols]
y = df["travel_style"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

label_encoder = LabelEncoder()
label_encoder.fit(y)
y_train_enc = label_encoder.transform(y_train)
y_test_enc = label_encoder.transform(y_test)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

pipelines = {
    "logreg": Pipeline(
        [
            ("prep", preprocess),
            ("clf", LogisticRegression(max_iter=400, random_state=42)),
        ]
    ),
    "rf": Pipeline(
        [
            ("prep", preprocess),
            ("clf", RandomForestClassifier(random_state=42)),
        ]
    ),
    "gbrt": Pipeline(
        [
            ("prep", preprocess),
            ("clf", GradientBoostingClassifier(random_state=42)),
        ]
    ),
}

param_grids = {
    "logreg": {"clf__C": [0.25, 1.0, 4.0]},
    "rf": {"clf__n_estimators": [120, 240], "clf__max_depth": [None, 12]},
    "gbrt": {"clf__learning_rate": [0.05, 0.1], "clf__n_estimators": [80, 160]},
}

best_name, best_score, best_pipe = None, -1.0, None
for name, pipe in pipelines.items():
    gs = GridSearchCV(pipe, param_grids[name], cv=3, n_jobs=-1)
    gs.fit(X_train, y_train_enc)
    score = float(gs.score(X_test, y_test_enc))
    print(name, "best params", gs.best_params_, "acc", score)
    if score > best_score:
        best_name, best_score, best_pipe = name, score, gs.best_estimator_

print("Winner:", best_name, best_score)
print(classification_report(y_test_enc, best_pipe.predict(X_test)))

out_dir = Path("../backend/ml/models")
out_dir.mkdir(parents=True, exist_ok=True)

best_pipe.fit(X, label_encoder.transform(y))

clf = best_pipe.named_steps["clf"]
prep = best_pipe.named_steps["prep"]
joblib.dump(clf, out_dir / "travel_classifier_final.joblib")
joblib.dump(prep, out_dir / "preprocessor.joblib")
joblib.dump(label_encoder, out_dir / "label_encoder.joblib")
joblib.dump({"classifier": str(out_dir / "travel_classifier_final.joblib")}, out_dir / "model.joblib")
print("Saved artifacts to", out_dir.resolve())